# Random Forest — Water Quality Targets

Trains a **Random Forest regression** model for each of twelve water-quality
target variables using the terminal modeling table
`data/03c_merge_tertiary/epa-full.csv`, then evaluates every model on a held-out
test split and reports **R²**, **RMSE**, and **Error Rate** (symmetric MAPE, %).

This is the tree-based counterpart to `multiple_linear_regression.ipynb` — the
targets, features, split, and metrics are identical, so the two notebooks are
directly comparable.

**Targets modeled:** water temperature, dissolved oxygen, pH, nitrate, nitrite,
nitrate + nitrite, total phosphorus, specific conductance, total dissolved
solids, total suspended solids, turbidity, and *E. coli*.

Each model is a scikit-learn `Pipeline`:

1. `SimpleImputer(strategy="median")` — fill missing predictor values
2. `RandomForestRegressor(...)` — the ensemble estimator

No feature scaling is needed: a random forest is invariant to monotone
rescaling of the inputs.

The notebook is split into parts:

* **Part 1 — Train** every model and keep its held-out test split.
* **Part 2 — Test** every trained model and summarize the three metrics.
* **Part 3 — Feature importances** for interpretability.


In [1]:
# --- Imports ---
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
RANDOM_STATE = 42
TEST_SIZE = 0.2
MIN_SAMPLES = 100  # skip a target with fewer usable rows than this

# Random forest hyperparameters (shared across all targets).
RF_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

## Configuration

Locate the dataset, and declare the targets and predictor features.

In [2]:
# --- Locate the repo root and the terminal modeling table ---
# The notebook may be launched from anywhere; walk upward until we find the CSV.
def find_data_path() -> Path:
    rel = Path("data/03c_merge_tertiary/epa-full.csv")
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidate = base / rel
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {rel} by walking up from {here}. "
        "Run the notebook from within the repository."
    )

DATA_PATH = find_data_path()
print("Using dataset:", DATA_PATH)

Using dataset: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/03c_merge_tertiary/epa-full.csv


In [3]:
# --- Target variables: label -> (CSV column, plausible valid range) ---
# The valid range drops physically impossible readings and data-entry errors
# before fitting (e.g. a pH of 999 or a negative concentration).
TARGETS = {
    "Water Temperature":      {"column": "Temperature, water_value",            "valid_range": (-5.0, 45.0)},
    "Dissolved Oxygen":       {"column": "Dissolved oxygen (DO)_value",         "valid_range": (0.0, 30.0)},
    "pH":                     {"column": "pH_value",                            "valid_range": (0.0, 14.0)},
    "Nitrate":                {"column": "Nitrate_value",                       "valid_range": (0.0, 100.0)},
    "Nitrite":                {"column": "Nitrite_value",                       "valid_range": (0.0, 20.0)},
    "Nitrate + Nitrite":      {"column": "Nitrate + Nitrite_value",             "valid_range": (0.0, 100.0)},
    "Total Phosphorus":       {"column": "Total Phosphorus, mixed forms_value", "valid_range": (0.0, 25.0)},
    "Specific Conductance":   {"column": "Specific conductance_value",          "valid_range": (0.0, 10000.0)},
    "Total Dissolved Solids": {"column": "Total dissolved solids_value",        "valid_range": (0.0, 10000.0)},
    "Total Suspended Solids": {"column": "Total suspended solids_value",        "valid_range": (0.0, 10000.0)},
    "Turbidity":              {"column": "Turbidity_value",                     "valid_range": (0.0, 5000.0)},
    "E. coli":                {"column": "Escherichia coli_value",              "valid_range": (0.0, 1_000_000.0)},
}

# --- Predictor features ---
# Environmental / spatial / temporal drivers only. We deliberately exclude the
# other water-quality "_value" columns so a model never predicts one target
# from another measured target.
BASE_FEATURE_COLS = [
    # Location
    "LatitudeMeasure", "LongitudeMeasure",
    "distance_to_climate_station_km", "distance_to_streamflow_gauge_km",
    # PRISM climate normals at the observation
    "prism_tmax_c", "prism_tmin_c", "prism_ppt_mm", "prism_tdmean_c",
    # ISU station weather
    "isu_avg_wind_speed_kts", "isu_avg_rh", "isu_snow_in", "isu_snowd_in",
    "isu_max_feel_c", "isu_min_feel_c",
    # Hydrology
    "streamflow_discharge_cfs",
    # Soil
    "ksat_mean", "awc_mean",
    # Land cover
    "pct_corn", "pct_soybean", "pct_developed", "pct_forest", "pct_row_crops",
    # Nutrient loading context
    "npfert__n__total_kg", "npfert__p__total_kg",
    "npmanure__total__n_kg", "npmanure__total__p_kg",
]

# Temporal features engineered from the timestamp (added in the next cell).
TEMPORAL_FEATURE_COLS = ["doy", "doy_sin", "doy_cos", "obs_year"]

FEATURE_COLS = BASE_FEATURE_COLS + TEMPORAL_FEATURE_COLS
print(f"{len(FEATURE_COLS)} predictor features")

30 predictor features


## Load and prepare the data

Parse the timestamp and derive seasonal (day-of-year) features.

In [4]:
def load_dataset(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)

    # Derive temporal predictors from the activity timestamp. Build them as one
    # block and concat once, so we don't fragment the already-wide frame.
    ts = pd.to_datetime(df["ActivityStartDateTime"], errors="coerce")
    doy = ts.dt.dayofyear
    radians = 2.0 * np.pi * doy / 365.25  # cyclical: day 365 sits next to day 1
    temporal = pd.DataFrame({
        "doy": doy,
        "obs_year": ts.dt.year,
        "doy_sin": np.sin(radians),
        "doy_cos": np.cos(radians),
    }, index=df.index)
    return pd.concat([df, temporal], axis=1)


data = load_dataset(DATA_PATH)
print("Rows:", len(data), "| Columns:", data.shape[1])

# Sanity-check every declared predictor actually exists.
missing = [c for c in FEATURE_COLS if c not in data.columns]
assert not missing, f"Missing predictor columns: {missing}"
data[FEATURE_COLS].describe().T[["count", "mean", "std", "min", "max"]]

Rows: 48251 | Columns: 319


,count,mean,std,min,max
LatitudeMeasure,"48,251.0000",41.8428,0.7139,40.3871,43.5002
LongitudeMeasure,"48,251.0000",-93.0813,1.4052,-96.6325,-90.2010
distance_to_climate_station_km,"48,251.0000",21.5023,12.0707,0.2631,74.5673
distance_to_streamflow_gauge_km,"48,251.0000",5.0970,4.8408,0.0000,25.8260
prism_tmax_c,"46,602.0000",21.8390,9.8877,-20.4180,38.9453
prism_tmin_c,"46,632.0000",10.1485,9.2721,-29.3780,27.0850
prism_ppt_mm,"46,629.0000",3.3790,9.4365,0.0000,131.6350
prism_tdmean_c,"46,625.0000",10.8877,9.3835,-29.1303,27.2638
isu_avg_wind_speed_kts,"45,079.0000",7.0451,3.5216,0.0000,25.5105
isu_avg_rh,"44,806.0000",72.6710,12.8659,1.0227,100.0000


## Metrics

* **R²** — coefficient of determination on the held-out test set.
* **RMSE** — root mean squared error, in the target's own units.
* **Error Rate** — symmetric mean absolute percentage error (sMAPE), reported as
  a percentage. sMAPE is bounded and stays well-behaved when the true value is
  near zero, which matters for skewed concentration targets. This matches the
  `error_rate_pct` convention used elsewhere in the repo.


In [5]:
def symmetric_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Symmetric MAPE as a percentage (0 = perfect). Robust to y_true near 0."""
    denom = np.abs(y_true) + np.abs(y_pred)
    numer = 2.0 * np.abs(y_true - y_pred)
    safe = np.divide(numer, denom, out=np.zeros_like(denom, dtype=float), where=denom != 0)
    return float(np.mean(safe) * 100.0)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "Error Rate (%)": symmetric_mape(y_true, y_pred),
    }

## Part 1 — Train every model

For each target we drop rows with no measurement, clip to the valid range, split off a 20% test set, and fit the random-forest pipeline. Trained models and their splits are cached in `TRAINED` for the testing section below. (Training all twelve forests takes a couple of minutes.)

In [6]:
def make_pipeline() -> Pipeline:
    """Impute -> random forest."""
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(**RF_PARAMS)),
    ])


def prepare_xy(df: pd.DataFrame, column: str, valid_range: tuple) -> tuple:
    """Return (X, y) for one target after dropping NaN and out-of-range rows."""
    frame = df[FEATURE_COLS + [column]].dropna(subset=[column]).copy()
    lo, hi = valid_range
    frame = frame[frame[column].between(lo, hi)]
    X = frame[FEATURE_COLS].to_numpy(dtype=float)
    y = frame[column].to_numpy(dtype=float)
    return X, y


TRAINED: dict[str, dict] = {}

for label, cfg in TARGETS.items():
    X, y = prepare_xy(data, cfg["column"], cfg["valid_range"])
    if len(y) < MIN_SAMPLES:
        print(f"[SKIP] {label}: only {len(y)} usable rows (< {MIN_SAMPLES}).")
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    pipeline = make_pipeline()
    pipeline.fit(X_train, y_train)

    TRAINED[label] = {
        "column": cfg["column"],
        "model": pipeline,
        "X_test": X_test,
        "y_test": y_test,
        "n_total": len(y),
        "n_train": len(y_train),
        "n_test": len(y_test),
    }
    print(f"[OK]   {label:24s} trained on {len(y_train):>6,} rows "
          f"(test {len(y_test):>5,}) of {len(y):>6,} usable")

print(f"\nTrained {len(TRAINED)} of {len(TARGETS)} target models.")

[OK]   Water Temperature        trained on 27,764 rows (test 6,941) of 34,705 usable


[OK]   Dissolved Oxygen         trained on 25,460 rows (test 6,365) of 31,825 usable


[OK]   pH                       trained on 25,882 rows (test 6,471) of 32,353 usable


[OK]   Nitrate                  trained on  9,877 rows (test 2,470) of 12,347 usable


[OK]   Nitrite                  trained on  9,288 rows (test 2,323) of 11,611 usable


...

[OK]   Specific Conductance     trained on 12,853 rows (test 3,214) of 16,067 usable


[OK]   Total Dissolved Solids   trained on 14,408 rows (test 3,603) of 18,011 usable


[OK]   Total Suspended Solids   trained on 11,618 rows (test 2,905) of 14,523 usable


[OK]   Turbidity                trained on 16,924 rows (test 4,232) of 21,156 usable


[OK]   E. coli                  trained on 12,717 rows (test 3,180) of 15,897 usable

Trained 12 of 12 target models.


## Part 2 — Test every model

`test_model()` scores one cached model on its held-out test set and returns the three required metrics. `test_all_models()` runs it across every trained target and assembles a summary table.

In [7]:
def test_model(label: str, verbose: bool = True) -> dict:
    """Evaluate one trained model on its held-out test set."""
    if label not in TRAINED:
        raise KeyError(f"No trained model for {label!r}. Run Part 1 first.")
    entry = TRAINED[label]
    y_pred = entry["model"].predict(entry["X_test"])
    metrics = evaluate(entry["y_test"], y_pred)
    if verbose:
        print(f"{label}  (n_test={entry['n_test']:,})")
        print(f"    R2         = {metrics['R2']:.4f}")
        print(f"    RMSE       = {metrics['RMSE']:.4f}")
        print(f"    Error Rate = {metrics['Error Rate (%)']:.2f}%")
    return metrics


def test_all_models() -> pd.DataFrame:
    rows = []
    for label, entry in TRAINED.items():
        m = test_model(label, verbose=False)
        rows.append({
            "Target": label,
            "Column": entry["column"],
            "N test": entry["n_test"],
            "R2": m["R2"],
            "RMSE": m["RMSE"],
            "Error Rate (%)": m["Error Rate (%)"],
        })
    summary = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
    return summary

### Per-model report

In [8]:
for label in TRAINED:
    test_model(label)
    print()

Water Temperature  (n_test=6,941)
    R2         = 0.9517
    RMSE       = 1.8374
    Error Rate = 17.48%

Dissolved Oxygen  (n_test=6,365)
    R2         = 0.5962
    RMSE       = 1.6884
    Error Rate = 13.27%

pH  (n_test=6,471)
    R2         = 0.5261
    RMSE       = 0.4560
    Error Rate = 3.85%

Nitrate  (n_test=2,470)
    R2         = 0.6742
    RMSE       = 3.0372
    Error Rate = 103.22%

Nitrite  (n_test=2,323)
    R2         = 0.0863
    RMSE       = 0.1383
    Error Rate = 175.49%



Nitrate + Nitrite  (n_test=932)
    R2         = 0.6563
    RMSE       = 2.4244
    Error Rate = 59.00%

Total Phosphorus  (n_test=1,178)
    R2         = 0.3298
    RMSE       = 0.3566
    Error Rate = 57.04%



Specific Conductance  (n_test=3,214)
    R2         = 0.8864
    RMSE       = 67.2773
    Error Rate = 6.96%

Total Dissolved Solids  (n_test=3,603)
    R2         = 0.8243
    RMSE       = 52.6977
    Error Rate = 9.05%

Total Suspended Solids  (n_test=2,905)
    R2         = 0.3913
    RMSE       = 163.7680
    Error Rate = 78.83%

Turbidity  (n_test=4,232)
    R2         = 0.2750
    RMSE       = 102.9504
    Error Rate = 63.98%



E. coli  (n_test=3,180)
    R2         = 0.2601
    RMSE       = 6627.6175
    Error Rate = 115.81%



### Summary table

All twelve models side by side, sorted by test R².

In [9]:
summary = test_all_models()
summary

,Target,Column,N test,R2,RMSE,Error Rate (%)
0,Water Temperature,"Temperature, water_value",6941,0.9517,1.8374,17.4849
1,Specific Conductance,Specific conductance_value,3214,0.8864,67.2773,6.9578
2,Total Dissolved Solids,Total dissolved solids_value,3603,0.8243,52.6977,9.0535
3,Nitrate,Nitrate_value,2470,0.6742,3.0372,103.2159
4,Nitrate + Nitrite,Nitrate + Nitrite_value,932,0.6563,2.4244,58.9997
5,Dissolved Oxygen,Dissolved oxygen (DO)_value,6365,0.5962,1.6884,13.2715
6,pH,pH_value,6471,0.5261,0.4560,3.8452
7,Total Suspended Solids,Total suspended solids_value,2905,0.3913,163.7680,78.8335
8,Total Phosphorus,"Total Phosphorus, mixed forms_value",1178,0.3298,0.3566,57.0408
9,Turbidity,Turbidity_value,4232,0.2750,102.9504,63.9818


### Test a single model on demand

Change `target` to re-run the evaluation for any one model.

In [10]:
target = "Water Temperature"
_ = test_model(target)

Water Temperature  (n_test=6,941)
    R2         = 0.9517
    RMSE       = 1.8374
    Error Rate = 17.48%


## Part 3 — Feature importances

Unlike linear regression, a random forest exposes impurity-based feature importances. `top_features()` shows the strongest predictors for any target.

In [11]:
def top_features(label: str, n: int = 10) -> pd.DataFrame:
    """Return the n most important features for a trained model."""
    importances = TRAINED[label]["model"].named_steps["model"].feature_importances_
    return (
        pd.DataFrame({"feature": FEATURE_COLS, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


top_features("Water Temperature")

,feature,importance
0,prism_tmin_c,0.1909
1,prism_tdmean_c,0.1542
2,prism_tmax_c,0.1272
3,isu_min_feel_c,0.1237
4,doy_cos,0.1104
5,isu_max_feel_c,0.0814
6,doy,0.0690
7,doy_sin,0.0442
8,isu_avg_wind_speed_kts,0.0093
9,LongitudeMeasure,0.0083


## Part 4 — Save trained models

Persist every fitted pipeline to `src/05_modeling/random_forest/` as `rf_<target>.pkl`. Each file is self-contained (imputer + estimator) and can be reloaded with `pickle.load` for inference.

In [12]:
import pickle
import re

# Write .pkl files into src/05_modeling/random_forest/ regardless of launch dir.
REPO_ROOT = DATA_PATH.parents[2]
MODEL_DIR = REPO_ROOT / "src" / "05_modeling" / "random_forest"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

PREFIX = "rf"  # random forest

def target_stem(label: str) -> str:
    """'Nitrate + Nitrite' -> 'nitrate_nitrite', 'E. coli' -> 'e_coli'."""
    return re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_")

saved = []
for label, entry in TRAINED.items():
    path = MODEL_DIR / f"{PREFIX}_{target_stem(label)}.pkl"
    with open(path, "wb") as f:
        pickle.dump(entry["model"], f)
    saved.append(path.name)

print(f"Saved {len(saved)} models to {MODEL_DIR}:")
for name in saved:
    print("  ", name)

Saved 12 models to /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/src/05_modeling/random_forest:
   rf_water_temperature.pkl
   rf_dissolved_oxygen.pkl
   rf_ph.pkl
   rf_nitrate.pkl
   rf_nitrite.pkl
   rf_nitrate_nitrite.pkl
   rf_total_phosphorus.pkl
   rf_specific_conductance.pkl
   rf_total_dissolved_solids.pkl
   rf_total_suspended_solids.pkl
   rf_turbidity.pkl
   rf_e_coli.pkl


---

**Notes**

* Compared with `multiple_linear_regression.ipynb`, the random forest captures
  non-linear relationships and interactions, so it typically posts higher R² and
  lower error rates — especially for the skewed concentration targets (nitrate,
  turbidity, *E. coli*) where the linear baseline struggles.
* To persist a fitted model, `pickle.dump(TRAINED[label]["model"], ...)`; the
  pipeline is self-contained (imputer + estimator).
* `RF_PARAMS` at the top controls the forest. Raising `n_estimators` trades
  training time for a marginally steadier fit.
